# 02 - Ingeniería de Características

## Descripción General

La ingeniería de características (*feature Engineering*) es el proceso de transformar datos brutos en entradas significativas (características) que ayuden a los modelos de machine learning a aprender patrones de manera más efectiva y mejorar la precisión de las predicciones. Esto implica la creación de nuevas características, transformación de las características existentes, imputación de valores faltantes, codificación de datos categóricos y selección de las características más relevantes.

Dado que el rendimiento de un modelo depende en gran medida de la calidad de los datos de entrenamiento, la ingeniería de características se convierte en un paso crucial del preprocesamiento. Su objetivo es transformar y seleccionar los aspectos más relevantes de los datos brutos, alineándolos tanto con la tarea predictiva como con las necesidades específicas del modelo, para maximizar su capacidad de aprendizaje y generalización.

En esta etapa dejamos de “explorar” y empezamos a construir lo que el modelo realmente va a usar. Todo el EDA previo tuvo un propósito claro: identificar qué variables contienen señal y cuáles no. Aquí convertiremos ese análisis en un proceso reproducible.

Partiremos del conjunto de datos `df_features` que contiene las características candidatas al modelo y construiremos un conjunto de variables (X) que capturen las señales mas relevantes sobre los factores que influyen en la fijación de precios de los alojamientos de Airbnb en la Ciudad de México. Para lograrlo, se dividirá el proceso en dos niveles:

- **Creación de features (`build_features`):**  
  Generar nuevas variables a partir de los datos originales. Estas pueden incluir features o métricas derivadas, combinaciones de atributos o indicadores que aporten señales adicionales al modelo.

- **Transformación de features (`transform_features`):**  
  Modificar las variables existentes para optimizar su representación. Esto abarca operaciones como normalización, escalado, codificación de categorías o transformaciones matemáticas que mejoren la capacidad predictiva.

**Flujo completo**


```text
df_clean
   ↓
df_features
   ↓
build_features()
   ↓
transform_features()
   ↓
df_model → X listo para modelado

## Título 2

### Importación de librerías y carga del dataset

In [1]:
# Import libraries, modules and dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import sys
import os

# Add project root to path
sys.path.append(os.path.abspath(".."))

# Load the autoreload extension to automatically reload modules when they change
# Set autoreload mode to 2: reload all modules before executing user code
%load_ext autoreload
%autoreload 2 

# Import validate features module
from src.features.validate_feature import validate_features

# Load dataset
df_features = pd.read_csv("../data/processed/df_features.csv")

# Preview first rows
df_features.head()

,listing_url,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,...,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,number_of_reviews,price,clean_price,log_price,price_per_guest
0,https://www.airbnb.com/rooms/35797,"Mexico City, D.f., Mexico",Cuajimalpa de Morelos,NaN,19.38283,-99.27178,Entire villa,Entire home/apt,2,1.0,...,NaN,NaN,NaN,NaN,NaN,0,"$3,673.00",3673.0,8.208764,1836.500000
1,https://www.airbnb.com/rooms/44616,NaN,Cuauhtémoc,NaN,19.41162,-99.17794,Entire home,Entire home/apt,14,5.5,...,4.70,4.87,4.78,4.98,4.47,65,"$18,000.00",18000.0,9.798127,1285.714286
2,https://www.airbnb.com/rooms/56074,"Mexico City, DF, Mexico",Cuauhtémoc,NaN,19.43977,-99.15605,Entire condo,Entire home/apt,2,1.0,...,4.88,4.98,4.94,4.76,4.79,84,$591.00,591.0,6.381816,295.500000
3,https://www.airbnb.com/rooms/165772,"Mexico City, Federal District, Mexico",Miguel Hidalgo,NaN,19.40826,-99.18659,Entire home,Entire home/apt,16,5.0,...,4.84,4.91,4.89,4.75,4.90,386,"$3,673.00",3673.0,8.208764,229.562500
4,https://www.airbnb.com/rooms/171109,"Mexico City, Federal District, Mexico",Benito Juárez,NaN,19.39675,-99.17581,Private room in rental unit,Private room,2,1.0,...,4.61,4.98,4.95,4.97,4.83,123,$321.00,321.0,5.771441,160.500000


## Creación de Características (Build Features)

En esta sección se construirán las variables derivadas que constituirán el conjunto de entrada (X) para el modelo. A partir de las variables originales, se generan nuevas features con capacidad explicativa, incorporando relaciones, agregaciones y transformaciones identificadas durante el EDA.

El objetivo no es aumentar el número de variables, sino capturar mejor la señal relevante: ubicación, características del listing, calidad percibida y contexto del entorno. Estas features se desarrollaran de forma modular para facilitar su validación, reutilización y posterior integración en el pipeline de modelado.

## Features de ubicación (Geo)

La ubicación es uno de los factores más determinantes en el precio de un listing. Sin embargo, variables como latitud y longitud por sí solas no son directamente interpretables por el modelo. En esta sección se construirán features geoespaciales que capturan accesibilidad y atractivo del entorno, como la cercanía a zonas turísticas o culturales y la densidad de actividad comercial en la zona. Con estas variables buscaremos traducir la ubicación en señales más informativas y comparables entre listings.

In [2]:
# Import the function to add geo features
from src.features.geo import add_distance_to_attractions
from src.features.geo import add_attractions_density
from src.features.geo import add_commercial_density

# Apply the function to df_features to add the new columns
df_features = add_distance_to_attractions(df_features)
df_features = add_attractions_density(df_features)
df_features = add_commercial_density(df_features)

# Display the first rows of latitude, longitude, and the new geo features
geo_features = ['dist_to_nearest_attraction', 'attractions_within_radius', 'commercial_within_radius']
df_features[["latitude", "longitude"] + geo_features]


,latitude,longitude,dist_to_nearest_attraction,attractions_within_radius,commercial_within_radius
0,19.382830,-99.271780,2.450990,0,1
1,19.411620,-99.177940,0.415358,4,252
2,19.439770,-99.156050,0.375601,3,129
3,19.408260,-99.186590,0.985700,1,53
4,19.396750,-99.175810,1.557305,0,116
...,...,...,...,...,...
21490,19.442240,-99.113440,2.178589,0,14
21491,19.308017,-99.168158,0.958782,1,29
21492,19.434460,-99.174010,0.832345,1,128
21493,19.406435,-99.160934,0.958943,1,184


In [3]:
# Validate geo_features
validate_features(df_features, geo_features, 'log_price')


========== Validating: dist_to_nearest_attraction ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0   float64  21495     21495   18081     0           0.0

--- Statistical Summary ---
                          0
mean               1.302349
median             0.716410
mode               0.606966
std                1.588781
min                0.004748
p25                0.394962
p50                0.716410
p75                1.572062
max               16.244343
skew               2.974055
kurt              11.983244
outliers_count  1837.000000
outliers_pct       8.550000
pearson_r         -0.239113

========== Validating: attractions_within_radius ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0     int64  21495     21495      12     0           0.0

--- Statistical Summary ---
                        0
mean             2.717562
median           2.000000
mode             0.000000
std              2.79

## Features de tipo de propiedad

In [4]:
# Import the function to add property features
from src.features.property import add_property_group
from src.features.property import add_property_group_room

# Apply the function to df_features to add the new columns
df_features = add_property_group(df_features)
df_features = add_property_group_room(df_features)

# Display the first rows of property features
property_features = ['room_type','property_group', 'property_group_room']
df_features[property_features]

,room_type,property_group,property_group_room
0,entire_home/apt,house,house_entire_home/apt
1,entire_home/apt,house,house_entire_home/apt
2,entire_home/apt,apartment,apartment_entire_home/apt
3,entire_home/apt,house,house_entire_home/apt
4,private_room,apartment,apartment_private_room
...,...,...,...
21490,private_room,apartment,apartment_private_room
21491,private_room,apartment,apartment_private_room
21492,entire_home/apt,apartment,apartment_entire_home/apt
21493,entire_home/apt,apartment,apartment_entire_home/apt


In [5]:
# Validate property_features
validate_features(df_features, property_features, 'log_price')


========== Validating: room_type ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0       str  21495     21495       4     0           0.0

--- Frequency distribution ---
                 count    pct
room_type                    
entire_home/apt  14783  68.77
private_room      6433  29.93
shared_room        232   1.08
hotel_room          47   0.22

--- Median target by category ---
         room_type  median_target
0       hotel_room       7.600902
1  entire_home/apt       7.133296
2     private_room       6.363028
3      shared_room       5.579730

========== Validating: property_group ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0       str  21495     21495       6     0           0.0

--- Frequency distribution ---
                count    pct
property_group              
apartment       16604  77.25
house            2881  13.40
hotel             992   4.62
guesthouse        804   3.74
unique/n

## Features de restricciones de reserva

In [8]:
# Import the function to add booking restrictions features
from src.features.booking_restrictions import add_minimum_nights_segment

# Apply the function to df_features to add the new column
df_features = add_minimum_nights_segment(df_features)

# Display the first rows of booking restrictions features
booking_restrictions_features = ['minimum_nights', 'minimum_nights_segment']
df_features[booking_restrictions_features]

,minimum_nights,minimum_nights_segment
0,1,short_stay
1,1,short_stay
2,15,medium_stay
3,2,short_stay
4,4,medium_stay
...,...,...
21490,1,short_stay
21491,1,short_stay
21492,1,short_stay
21493,1,short_stay


In [9]:
validate_features(df_features, booking_restrictions_features, 'log_price')


========== Validating: minimum_nights ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0     int64  21495     21495      52     0           0.0

--- Statistical Summary ---
                          0
mean               3.171110
median             1.000000
mode               1.000000
std               13.396934
min                1.000000
p25                1.000000
p50                1.000000
p75                2.000000
max              365.000000
skew              20.920417
kurt             522.484631
outliers_count  2180.000000
outliers_pct      10.140000
pearson_r          0.029767

========== Validating: minimum_nights_segment ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0  category  21495     21495       3     0           0.0

--- Frequency distribution ---
                        count    pct
minimum_nights_segment              
short_stay              19315  89.86
medium_stay              1

## Features de amenidades y servicios

### Creación de `amenity_count` y `amenity_count_binned`

In [ ]:
# Import the function to add amenities features
from src.features.amenities import add_amenity_count
from src.features.amenities import add_amenity_count_binned

# Apply the function to df_features to add the new columns
df_features = add_amenity_count(df_features)
df_features = add_amenity_count_binned(df_features)

# Display the first rows of amenities features
amenity_features = ['amenity_count', 'amenity_count_binned']
df_features[amenity_features]

,amenity_count,amenity_count_binned
0,12,low
1,26,low
2,28,medium
3,47,high
4,24,low
...,...,...
21490,28,medium
21491,9,low
21492,17,low
21493,11,low


In [ ]:
validate_features(df_features, amenity_features, 'log_price')


========== Validating: amenity_count ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0     int64  21495     21495      92     0           0.0

--- Statistical Summary ---
                         0
mean             33.147383
median           34.000000
mode             35.000000
std              15.537351
min               0.000000
p25              22.000000
p50              34.000000
p75              44.000000
max             102.000000
skew              0.070188
kurt             -0.428441
outliers_count   68.000000
outliers_pct      0.320000
pearson_r         0.335379

========== Validating: amenity_count_binned ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0  category  21495     21495       3     0           0.0

--- Frequency distribution ---
                      count    pct
amenity_count_binned              
medium                 7560  35.17
low                    7242  33.69
high           

### Creación de columnas binarias `has_amenities`

In [ ]:
# Import the function to add has_amenity features
from src.features.amenities import add_has_amenity_features

# Apply the function to df_features to add the new columns
df_features, has_amenity_cols = add_has_amenity_features(df_features)

# Display the first rows of has_amenity columns
df_features[has_amenity_cols]

,has_wifi,has_kitchen,has_hot_water,has_essentials,has_bed_linens,has_microwave,has_refrigerator,has_air_conditioning,has_heating,has_cooking_basics,...,has_backyard,has_sauna,has_city_skyline_view,has_outdoor_furniture,has_outdoor_dining_area,has_smoke_alarm,has_carbon_monoxide_alarm,has_fire_extinguisher,has_exterior_security_cameras_on_property,has_first_aid_kit
0,True,True,True,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,True,True,False,True,False,True,True,False,False,True,...,True,False,False,False,False,True,True,False,True,False
2,True,True,True,True,True,True,True,False,False,True,...,False,False,False,False,False,False,False,False,False,False
3,True,True,True,True,True,True,True,False,True,True,...,True,False,False,True,True,False,False,False,False,True
4,True,True,True,True,False,True,True,False,False,True,...,False,False,False,False,False,False,False,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21490,True,True,True,False,True,True,True,False,False,True,...,False,False,False,False,False,True,True,False,True,False
21491,True,True,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
21492,True,True,True,True,True,True,True,False,False,False,...,False,False,False,False,False,True,True,False,False,False
21493,True,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,True,True,True,True


In [ ]:
# Compute mean occurrence rate of each has_amenity (proportion of listings with the amenity)
df_features[has_amenity_cols].mean().sort_values(ascending=False)

has_wifi                                     0.990463
has_kitchen                                  0.892021
has_tv                                       0.860526
has_hot_water                                0.844010
has_cooking_basics                           0.750686
has_dishes_and_silverware                    0.743196
has_iron                                     0.742266
has_essentials                               0.717097
has_dedicated_workspace                      0.701838
has_bed_linens                               0.698488
has_refrigerator                             0.693929
has_coffee_maker                             0.681600
has_microwave                                0.677786
has_carbon_monoxide_alarm                    0.657967
has_hair_dryer                               0.635729
has_self_check_in                            0.588183
has_smoke_alarm                              0.578507
has_dining_table                             0.493557
has_first_aid_kit           

### Creación de `amenity_score`

In [ ]:
# Import the function to add amenity_score feature
from src.features.amenities import add_amenity_score

# Apply the function to df_features to add the new columns
df_features = add_amenity_score(df_features)

# Display the first rows of has_amenities columns
df_features['amenity_score']

0        0.268380
1        1.044497
2        0.976673
3        1.261502
4        1.033924
           ...   
21490    0.649838
21491    0.409493
21492    0.693755
21493    0.286073
21494    0.286073
Name: amenity_score, Length: 21495, dtype: float64

In [ ]:
validate_features(df_features, ['amenity_score'], 'log_price')


========== Validating: amenity_score ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0   float64  21495     21495   11704     0           0.0

--- Statistical Summary ---
                       0
mean            0.937367
median          1.001418
mode            0.227788
std             0.388871
min             0.000000
p25             0.662696
p50             1.001418
p75             1.234661
max             1.874167
skew           -0.400404
kurt           -0.628622
outliers_count  0.000000
outliers_pct    0.000000
pearson_r       0.429282
